# HDP own-site adapter

Control panel for `hdp.run` (`1_scrapers/hdp/src/run.py`). All logic lives in the module; this notebook only calls it. Every path comes from `core.paths`.

**What it does.** One POST to the location API behind holdeineplakette.de/hauptuntersuchung (`getAllPublicLocations`, body `{"data":{}}`, no key). One row per location `id` (the `slugs` are SEO aliases, never separate sites). The display name is never used for matching: street, number, postcode and coordinates are the identity of a site.

**Hard-fail.** The request fails after its retries, `result.success` is not true, the locations list is missing or empty, or any location lacks an id, a postcode or coordinates. The number of sites is **not** a pass condition: a count that differs from the last passing full run is only a warning.

**Consumer.** The aggregator (`core.hdp_removal`) reads the newest HDP run, which must be a PASS and at most 7 days old, and removes the matching KÜS rows with a strict N == R check. Run the scraper right before the aggregator.

| Mode | Requests | Goes to |
|---|---|---|
| `simulated` | none (fake API) | `test_output/` |
| `full` | live, 1 POST; needs `confirm_full_run=True` | `output/` |
| `rebuild` | none, re-parses the `raw.jsonl` of an earlier run folder | `output/` |

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from core import paths
from hdp import run as hdp

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
ROOT = paths.project_root()

## 1. Simulated run (offline)

In [ ]:
res = hdp.run("simulated")
print(res.outcome)
res.frame[["location_id", "street", "zip", "city", "lat", "lng"]]

## 2. Full run (live, ONE request). Only when you want it.

In [ ]:
confirm_full_run = False  # set to True on purpose
if confirm_full_run:
    res = hdp.run("full", confirm_full_run=True)
    print(res.outcome, "| sites:", len(res.frame), "| warnings:", res.summary["warnings"])
    display(res.frame[["location_id", "name", "street", "zip", "city", "lat", "lng", "doubt_flags"]])
else:
    print("not started (confirm_full_run is False)")